<a href="https://colab.research.google.com/github/bojannithya-tech/nithyabojan/blob/main/AI_POS_Transaction_Anomaly_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Synthetic POS Transaction Dataset

Part 1A — Create the first 3 POS transactions

In [ ]:
# Synthetic POS transaction dataset
# We start with 3 scenarios:
# 1. Normal cashback
# 2. Higher-than-expected cashback
# 3. Negative cashback

transactions = [
    {
        "transaction_id": "TXN001",
        "store_id": "STORE101",
        "lane_id": "LANE01",

        "items": [
            {"item_id": "ITEM001", "price": 20.00, "quantity": 2},
            {"item_id": "ITEM002", "price": 10.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },

    {
        "transaction_id": "TXN002",
        "store_id": "STORE101",
        "lane_id": "LANE02",

        "items": [
            {"item_id": "ITEM101", "price": 40.00, "quantity": 1},
            {"item_id": "ITEM102", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": True,
            "amount": 25.00,
            "activation_status": "FAILED",
            "failure_amount": 25.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 70.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    },

    {
        "transaction_id": "TXN003",
        "store_id": "STORE102",
        "lane_id": "LANE03",

        "items": [
            {"item_id": "ITEM201", "price": 50.00, "quantity": 1},
            {"item_id": "ITEM202", "price": 20.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 10.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 60.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": -15.00
        }
    }
]

print("Synthetic transactions created:", len(transactions))

Synthetic transactions created: 3


Part 1B — Display the transactions

In [ ]:
import json

for transaction in transactions:
    print("=" * 60)
    print(json.dumps(transaction, indent=2))

{
  "transaction_id": "TXN001",
  "store_id": "STORE101",
  "lane_id": "LANE01",
  "items": [
    {
      "item_id": "ITEM001",
      "price": 20.0,
      "quantity": 2
    },
    {
      "item_id": "ITEM002",
      "price": 10.0,
      "quantity": 1
    }
  ],
  "basket_total": 50.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": false,
    "amount": 0.0,
    "activation_status": "NOT_APPLICABLE",
    "failure_amount": 0.0
  },
  "payment": {
    "type": "DEBIT",
    "amount": 45.0,
    "status": "SUCCESS"
  },
  "cashback": {
    "expected": 5.0,
    "actual": 5.0
  }
}
{
  "transaction_id": "TXN002",
  "store_id": "STORE101",
  "lane_id": "LANE02",
  "items": [
    {
      "item_id": "ITEM101",
      "price": 40.0,
      "quantity": 1
    },
    {
      "item_id": "ITEM102",
      "price": 30.0,
      "quantity": 1
    }
  ],
  "basket_total": 70.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": true,
    "amount": 25.0,


Part 2 — Build the Transaction Health Check

## Part 2: Transaction Health Check

The Transaction Health Check performs deterministic validation
before invoking GenAI agents.

It classifies a completed POS transaction as:

- NORMAL
- HIGH_CASHBACK
- NEGATIVE_CASHBACK

Financial calculations are performed using deterministic Python
logic rather than relying on an LLM.

In [ ]:
def check_transaction_health(transaction):

    expected = transaction["cashback"]["expected"]
    actual = transaction["cashback"]["actual"]

    variance = round(actual - expected, 2)

    # Negative cashback has highest priority
    if actual < 0:
        status = "ANOMALY"
        anomaly_type = "NEGATIVE_CASHBACK"

    # Cashback greater than expected
    elif actual > expected:
        status = "ANOMALY"
        anomaly_type = "HIGH_CASHBACK"

    else:
        status = "NORMAL"
        anomaly_type = "NONE"

    return {
        "transaction_id": transaction["transaction_id"],
        "status": status,
        "anomaly_type": anomaly_type,
        "expected_cashback": expected,
        "actual_cashback": actual,
        "variance": variance
    }

Part 2B — Test the Health Check

In [ ]:
health_results = []

for transaction in transactions:

    result = check_transaction_health(transaction)
    health_results.append(result)

    print("=" * 50)
    print("Transaction ID :", result["transaction_id"])
    print("Status         :", result["status"])
    print("Anomaly Type   :", result["anomaly_type"])
    print("Expected       : $", result["expected_cashback"])
    print("Actual         : $", result["actual_cashback"])
    print("Variance       : $", result["variance"])

Transaction ID : TXN001
Status         : NORMAL
Anomaly Type   : NONE
Expected       : $ 5.0
Actual         : $ 5.0
Variance       : $ 0.0
Transaction ID : TXN002
Status         : ANOMALY
Anomaly Type   : HIGH_CASHBACK
Expected       : $ 5.0
Actual         : $ 30.0
Variance       : $ 25.0
Transaction ID : TXN003
Status         : ANOMALY
Anomaly Type   : NEGATIVE_CASHBACK
Expected       : $ 5.0
Actual         : $ -15.0
Variance       : $ -20.0


Part 3 — Pattern & Correlation Analysis

## Part 3: Pattern and Correlation Analysis

For anomalous transactions, the system investigates whether the
cashback variance correlates with other monetary values in the
transaction.

The analysis checks relationships with:

- Gift card amount
- Gift card failure amount
- Promotion/discount amount
- Payment amount
- Individual item amounts

The detected correlations are treated as investigation evidence,
not as confirmed root causes.

Create the correlation tool

In [ ]:
def analyze_amount_correlations(transaction, health_result):

    # Normal transactions don't require deeper investigation
    if health_result["status"] == "NORMAL":
        return {
            "transaction_id": transaction["transaction_id"],
            "correlations": [],
            "message": "No anomaly detected. Correlation analysis not required."
        }

    expected = health_result["expected_cashback"]
    actual = health_result["actual_cashback"]

    # For HIGH cashback, investigate the excess.
    # For NEGATIVE cashback, investigate the magnitude of the negative value.
    if health_result["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(actual - expected, 2)
        investigation_basis = "EXCESS_CASHBACK"

    elif health_result["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(abs(actual), 2)
        investigation_basis = "NEGATIVE_CASHBACK_MAGNITUDE"

    else:
        suspicious_amount = 0
        investigation_basis = "NONE"

    correlations = []

    # Candidate transaction values
    candidates = {
        "gift_card_amount": transaction["gift_card"]["amount"],
        "gift_card_failure_amount": transaction["gift_card"]["failure_amount"],
        "promotion_discount": transaction["promotion"]["discount_amount"],
        "payment_amount": transaction["payment"]["amount"]
    }

    # Compare suspicious amount with transaction-level values
    for name, value in candidates.items():
        if value > 0 and abs(suspicious_amount - value) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": name,
                "value": value
            })

    # Compare against individual item amounts
    for item in transaction["items"]:

        item_total = round(item["price"] * item["quantity"], 2)

        if abs(suspicious_amount - item_total) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": f'item_total_{item["item_id"]}',
                "value": item_total
            })

    return {
        "transaction_id": transaction["transaction_id"],
        "anomaly_type": health_result["anomaly_type"],
        "investigation_basis": investigation_basis,
        "suspicious_amount": suspicious_amount,
        "correlations": correlations
    }

Run correlation analysis

In [ ]:
correlation_results = []

for transaction, health_result in zip(transactions, health_results):

    result = analyze_amount_correlations(
        transaction,
        health_result
    )

    correlation_results.append(result)

    print("=" * 60)
    print("Transaction:", transaction["transaction_id"])

    if health_result["status"] == "NORMAL":
        print("No anomaly - investigation skipped.")
        continue

    print("Anomaly:", result["anomaly_type"])
    print("Investigation Basis:", result["investigation_basis"])
    print("Suspicious Amount: $", result["suspicious_amount"])

    if result["correlations"]:
        print("\nCorrelations found:")

        for correlation in result["correlations"]:
            print(
                "  ->",
                correlation["field"],
                "= $",
                correlation["value"],
                "|",
                correlation["type"]
            )
    else:
        print("\nNo direct amount correlation found.")

Transaction: TXN001
No anomaly - investigation skipped.
Transaction: TXN002
Anomaly: HIGH_CASHBACK
Investigation Basis: EXCESS_CASHBACK
Suspicious Amount: $ 25.0

Correlations found:
  -> gift_card_amount = $ 25.0 | EXACT_MATCH
  -> gift_card_failure_amount = $ 25.0 | EXACT_MATCH
Transaction: TXN003
Anomaly: NEGATIVE_CASHBACK
Investigation Basis: NEGATIVE_CASHBACK_MAGNITUDE
Suspicious Amount: $ 15.0

No direct amount correlation found.


Add correlation strength

In [ ]:
def calculate_evidence_strength(correlation_result):

    correlations = correlation_result.get("correlations", [])

    if len(correlations) >= 2:
        return "STRONG"

    elif len(correlations) == 1:
        return "MODERATE"

    else:
        return "INSUFFICIENT"

In [ ]:
for result in correlation_results:

    if "correlations" not in result:
        continue

    strength = calculate_evidence_strength(result)

    result["evidence_strength"] = strength

    print(
        result["transaction_id"],
        "-> Evidence Strength:",
        strength
    )

TXN001 -> Evidence Strength: INSUFFICIENT
TXN002 -> Evidence Strength: STRONG
TXN003 -> Evidence Strength: INSUFFICIENT


Part 4 — Historical Transaction Context Analysis

## Part 4: Historical Transaction Context Analysis

Some rare POS anomalies may depend on transaction state carried across
transaction boundaries.

The Historical Context Analyzer examines previous transactions from the
same lane and checks whether earlier failure-related amounts correlate
with the current cashback anomaly.

This analysis generates investigation evidence only. A detected
correlation does not by itself prove causation.

Add a historical edge-case sequence

In [ ]:
historical_transactions = [
    {
        "transaction_id": "TXN004",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 1,

        "items": [
            {"item_id": "ITEM301", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 30.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 10.00,
            "activation_status": "FAILED",
            "failure_amount": 10.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 30.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN005",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 2,

        "items": [
            {"item_id": "ITEM302", "price": 40.00, "quantity": 1}
        ],

        "basket_total": 40.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 15.00,
            "activation_status": "FAILED",
            "failure_amount": 15.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 40.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN006",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 3,

        "items": [
            {"item_id": "ITEM303", "price": 50.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    }
]

print("Historical transactions created:", len(historical_transactions))

Historical transactions created: 3


4B — Historical Context Analyzer

In [ ]:
def analyze_historical_context(current_transaction, transaction_history):

    current_health = check_transaction_health(current_transaction)

    # Historical analysis is only required for anomalies
    if current_health["status"] == "NORMAL":
        return {
            "transaction_id": current_transaction["transaction_id"],
            "historical_correlation": False,
            "message": "Normal transaction - historical analysis not required."
        }

    # Determine suspicious amount
    if current_health["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(
            current_health["actual_cashback"]
            - current_health["expected_cashback"],
            2
        )

    elif current_health["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(
            abs(current_health["actual_cashback"]),
            2
        )

    else:
        suspicious_amount = 0.0

    # Only compare previous transactions from same store/lane
    relevant_history = [
        tx for tx in transaction_history
        if tx["store_id"] == current_transaction["store_id"]
        and tx["lane_id"] == current_transaction["lane_id"]
        and tx.get("sequence", 0) < current_transaction.get("sequence", 0)
    ]

    previous_failure_amounts = [
        tx["gift_card"]["failure_amount"]
        for tx in relevant_history
        if tx["gift_card"]["failure_amount"] > 0
    ]

    accumulated_failure_amount = round(
        sum(previous_failure_amounts),
        2
    )

    historical_match = (
        accumulated_failure_amount > 0
        and abs(suspicious_amount - accumulated_failure_amount) < 0.01
    )

    return {
        "transaction_id": current_transaction["transaction_id"],
        "anomaly_type": current_health["anomaly_type"],
        "suspicious_amount": suspicious_amount,
        "previous_failure_amounts": previous_failure_amounts,
        "accumulated_failure_amount": accumulated_failure_amount,
        "historical_correlation": historical_match
    }

Investigate TXN006

In [ ]:
current_transaction = historical_transactions[2]

history_result = analyze_historical_context(
    current_transaction,
    historical_transactions
)

print("Transaction:", history_result["transaction_id"])
print("Anomaly Type:", history_result["anomaly_type"])
print("Suspicious Cashback Amount: $", history_result["suspicious_amount"])

print(
    "Previous Gift Card Failure Amounts:",
    history_result["previous_failure_amounts"]
)

print(
    "Accumulated Previous Failure Amount: $",
    history_result["accumulated_failure_amount"]
)

print(
    "Historical Correlation:",
    history_result["historical_correlation"]
)

Transaction: TXN006
Anomaly Type: HIGH_CASHBACK
Suspicious Cashback Amount: $ 25.0
Previous Gift Card Failure Amounts: [10.0, 15.0]
Accumulated Previous Failure Amount: $ 25.0
Historical Correlation: True


Generate structured evidence

In [ ]:
def build_historical_evidence(history_result):

    if history_result.get("historical_correlation"):

        return {
            "evidence_type": "CROSS_TRANSACTION_CORRELATION",
            "strength": "STRONG",
            "observation": (
                f"The suspicious cashback amount of "
                f"${history_result['suspicious_amount']:.2f} "
                f"matches the accumulated previous gift-card "
                f"failure amount of "
                f"${history_result['accumulated_failure_amount']:.2f}."
            ),
            "interpretation": (
                "This may indicate that failure-related state "
                "persisted across transaction boundaries. "
                "Further investigation is required."
            )
        }

    return {
        "evidence_type": "NO_HISTORICAL_CORRELATION",
        "strength": "INSUFFICIENT",
        "observation": "No matching historical amount pattern was identified.",
        "interpretation": "No historical root-cause hypothesis can be supported."
    }


historical_evidence = build_historical_evidence(history_result)

print(json.dumps(historical_evidence, indent=2))

{
  "evidence_type": "CROSS_TRANSACTION_CORRELATION",
  "strength": "STRONG",
  "observation": "The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.",
  "interpretation": "This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required."
}


Part 5 — RAG Knowledge Base

## Part 5: RAG-Based POS Knowledge Retrieval

The investigation assistant uses Retrieval-Augmented Generation (RAG)
to retrieve relevant POS business and troubleshooting knowledge.

The knowledge base contains synthetic guidance related to:

- Cashback processing
- Gift card failure handling
- Transaction state management
- Negative cashback investigation
- Escalation procedures

RAG helps ground the investigation in retrieved evidence rather than
allowing the LLM to generate unsupported explanations.

Create the synthetic knowledge base

In [ ]:
knowledge_documents = [
    {
        "doc_id": "KB001",
        "title": "POS Cashback Processing Guide",
        "content": """
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
"""
    },

    {
        "doc_id": "KB002",
        "title": "Gift Card Failure Handling Guide",
        "content": """
Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
"""
    },

    {
        "doc_id": "KB003",
        "title": "POS Transaction State Management Guide",
        "content": """
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction monetary correlation should be treated as a
root-cause hypothesis requiring engineering validation.
"""
    },

    {
        "doc_id": "KB004",
        "title": "Negative Cashback Investigation Guide",
        "content": """
A negative cashback value should be treated as an anomaly and requires
investigation.

The investigation should examine adjustments, reversals, promotions,
refund-related values and tender events that may correlate with the
magnitude of the negative cashback.

If no supporting transaction evidence is available, the system should
not infer a root cause and should recommend manual investigation.
"""
    },

    {
        "doc_id": "KB005",
        "title": "POS Anomaly Escalation Guide",
        "content": """
An automated investigation should distinguish between observations,
correlations and confirmed root causes.

When evidence is insufficient or conflicting, the investigation result
should be marked as inconclusive.

Inconclusive anomalies should be escalated for manual engineering
investigation.

AI-generated hypotheses must remain subject to human review before
being accepted as a root-cause conclusion.
"""
    }
]

print("Knowledge documents created:", len(knowledge_documents))

Knowledge documents created: 5


Install embedding/vector-search libraries

In [ ]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.1 MB/s eta 0:00:00


Load the embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Create document embeddings

In [ ]:
document_texts = [
    doc["title"] + "\n" + doc["content"]
    for doc in knowledge_documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True
)

print("Number of documents:", len(document_texts))
print("Embedding shape:", document_embeddings.shape)

Number of documents: 5
Embedding shape: (5, 384)


Build the FAISS vector database

In [ ]:
dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(
    document_embeddings.astype("float32")
)

print("Documents stored in FAISS:", faiss_index.ntotal)

Documents stored in FAISS: 5


Build the Retriever

In [ ]:
def retrieve_pos_knowledge(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):

        document = knowledge_documents[index]

        results.append({
            "doc_id": document["doc_id"],
            "title": document["title"],
            "content": document["content"].strip(),
            "distance": float(distance)
        })

    return results

Test RAG retrieval

In [ ]:
query = """
A high cashback transaction has an excess amount that matches
accumulated gift card failure amounts from previous transactions.
What should be investigated?
"""

retrieved_documents = retrieve_pos_knowledge(
    query,
    top_k=2
)

for doc in retrieved_documents:

    print("=" * 70)
    print("Document:", doc["title"])
    print("Distance:", round(doc["distance"], 4))
    print()
    print(doc["content"])

Document: Gift Card Failure Handling Guide
Distance: 0.6626

Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
Document: POS Cashback Processing Guide
Distance: 0.8364

Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount cor

Connect the anomaly evidence to RAG

In [ ]:
def build_rag_query(health_result, historical_evidence):

    query = f"""
POS transaction anomaly investigation.

Anomaly type:
{health_result['anomaly_type']}

Expected cashback:
${health_result['expected_cashback']:.2f}

Actual cashback:
${health_result['actual_cashback']:.2f}

Cashback variance:
${health_result['variance']:.2f}

Historical evidence:
{historical_evidence['observation']}

Evidence interpretation:
{historical_evidence['interpretation']}

Retrieve POS knowledge that can help investigate this anomaly.
"""

    return query

In [ ]:
txn006_health = check_transaction_health(
    historical_transactions[2]
)

rag_query = build_rag_query(
    txn006_health,
    historical_evidence
)

print(rag_query)


POS transaction anomaly investigation.

Anomaly type:
HIGH_CASHBACK

Expected cashback:
$5.00

Actual cashback:
$30.00

Cashback variance:
$25.00

Historical evidence:
The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.

Evidence interpretation:
This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required.

Retrieve POS knowledge that can help investigate this anomaly.



In [ ]:
txn006_knowledge = retrieve_pos_knowledge(
    rag_query,
    top_k=2
)

for doc in txn006_knowledge:

    print("=" * 70)
    print(doc["title"])
    print("=" * 70)
    print(doc["content"])

POS Cashback Processing Guide
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
POS Transaction State Management Guide
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction moneta

Part 6 — GenAI Investigation Agent

## Part 6: GenAI Investigation Agent

The GenAI Investigation Agent combines:

- Transaction details
- Deterministic health-check results
- Current transaction correlations
- Historical transaction evidence
- Retrieved RAG knowledge

The agent generates an evidence-grounded investigation hypothesis and
recommended next steps.

The LLM does not perform the financial calculations and must not claim
a confirmed root cause when supporting evidence is insufficient.

Install Gemini SDK

In [ ]:
!pip -q install google-genai

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [ ]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY").strip()

print("Key loaded:", bool(api_key))
print("Contains newline:", "\n" in api_key)
print("Starts with AQ:", api_key.startswith("AQ"))

Key loaded: True
Contains newline: False
Starts with AQ: True


In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

print("Client created")

Client created


In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Reply with only: GEMINI CONNECTION SUCCESSFUL"
)

print(response.text)

GEMINI CONNECTION SUCCESSFUL


In [ ]:
def transaction_node_v2(state):

    transaction = state["transaction"]

    try:
        result = transaction_agent(transaction)

        log_agent(
            "Transaction Agent",
            transaction["transaction_id"],
            "Transaction parsed successfully"
        )

        return {"transaction_summary": result}

    except Exception as e:

        log_agent(
            "Transaction Agent",
            transaction.get("transaction_id", "UNKNOWN"),
            f"ERROR: {str(e)}"
        )

        return {
            "errors": [f"Transaction Agent Error: {str(e)}"]
        }

In [ ]:
def anomaly_node_v2(state):

    transaction = state["transaction"]

    try:
        result = anomaly_detection_agent(transaction)

        log_agent(
            "Anomaly Detection Agent",
            transaction["transaction_id"],
            f"Route selected: {result['route']}"
        )

        return {
            "health_result": result["health_result"],
            "route": result["route"]
        }

    except Exception as e:

        log_agent(
            "Anomaly Detection Agent",
            transaction.get("transaction_id", "UNKNOWN"),
            f"ERROR: {str(e)}"
        )

        return {
            "errors": [f"Anomaly Agent Error: {str(e)}"],
            "route": "MANUAL_REVIEW_PATH"
        }

Pattern and History nodes

In [ ]:
def pattern_node_v2(state):

    transaction = state["transaction"]

    try:
        result = pattern_analysis_agent(
            transaction,
            state["health_result"]
        )

        log_agent(
            "Pattern Analysis Agent",
            transaction["transaction_id"],
            f"{len(result.get('correlations', []))} correlation(s) found"
        )

        return {"pattern_evidence": result}

    except Exception as e:

        log_agent(
            "Pattern Analysis Agent",
            transaction["transaction_id"],
            f"ERROR: {str(e)}"
        )

        return {
            "pattern_evidence": {},
            "errors": [f"Pattern Agent Error: {str(e)}"]
        }

In [ ]:
def history_node_v2(state):

    transaction = state["transaction"]

    try:
        result = historical_context_agent(
            transaction,
            all_history
        )

        log_agent(
            "Historical Context Agent",
            transaction["transaction_id"],
            f"Historical correlation: "
            f"{result.get('analysis', {}).get('historical_correlation', False)}"
        )

        return {"historical_evidence": result}

    except Exception as e:

        log_agent(
            "Historical Context Agent",
            transaction["transaction_id"],
            f"ERROR: {str(e)}"
        )

        return {
            "historical_evidence": {},
            "errors": [f"Historical Agent Error: {str(e)}"]
        }

Create a join node

In [ ]:
def evidence_join_node(state):

    transaction_id = state["transaction"]["transaction_id"]

    log_agent(
        "Evidence Aggregator",
        transaction_id,
        "Pattern and historical evidence combined"
    )

    return {}

Build Version 2 LangGraph

In [ ]:
workflow_v2 = StateGraph(POSInvestigationState)

workflow_v2.add_node(
    "transaction_agent",
    transaction_node_v2
)

workflow_v2.add_node(
    "anomaly_agent",
    anomaly_node_v2
)

workflow_v2.add_node(
    "pattern_agent",
    pattern_node_v2
)

workflow_v2.add_node(
    "history_agent",
    history_node_v2
)

workflow_v2.add_node(
    "evidence_join",
    evidence_join_node
)

workflow_v2.add_node(
    "rag_agent",
    rag_node
)

workflow_v2.add_node(
    "rca_agent",
    rca_node
)

workflow_v2.add_node(
    "human_review",
    human_review_node
)

In [ ]:
workflow_v2.add_edge(
    START,
    "transaction_agent"
)

workflow_v2.add_edge(
    "transaction_agent",
    "anomaly_agent"
)

In [ ]:
workflow_v2.add_conditional_edges(
    "anomaly_agent",
    anomaly_router,
    {
        "normal": END,
        "investigate": "pattern_agent",
        "manual": END
    }
)

In [ ]:
workflow_v2.add_edge(
    "pattern_agent",
    "history_agent"
)

workflow_v2.add_edge(
    "history_agent",
    "evidence_join"
)

workflow_v2.add_edge(
    "evidence_join",
    "rag_agent"
)

workflow_v2.add_edge(
    "rag_agent",
    "rca_agent"
)

workflow_v2.add_edge(
    "rca_agent",
    "human_review"
)

workflow_v2.add_edge(
    "human_review",
    END
)

In [ ]:
pos_investigation_graph_v2 = workflow_v2.compile()

print("Version 2 workflow compiled successfully.")

Version 2 workflow compiled successfully.


One robustness test

In [ ]:
invalid_transaction = {
    "transaction_id": "TXN_BAD",
    "store_id": "STORE999",
    "lane_id": "LANE99"
}

In [ ]:
execution_log.clear()

bad_result = pos_investigation_graph_v2.invoke({
    "transaction": invalid_transaction,
    "errors": []
})

print(bad_result.get("errors"))

for entry in execution_log:
    print(
        entry["agent"],
        "->",
        entry["message"]
    )

["Anomaly Agent Error: 'cashback'"]
Transaction Agent -> ERROR: 'items'
Anomaly Detection Agent -> ERROR: 'cashback'


Part 10 — Evaluation